In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc matplotlib
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# トランスパイラーのステージ

<details>
<summary><b>Package versions</b></summary>

The code on this page was developed using the following requirements.
We recommend using these versions or newer.

```
qiskit[all]~=2.4.0
```
</details>

このページでは、Qiskit SDK に組み込まれたトランスパイルパイプラインのステージについて説明します。ステージは全部で六つあります。

1. `init`
2. `layout`
3. `routing`
4. `translation`
5. `optimization`
6. `scheduling`

[`generate_preset_pass_manager`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.generate_preset_pass_manager#qiskit.transpiler.generate_preset_pass_manager) 関数は、これらのステージから構成されるプリセットの[ステージ付きパスマネージャー](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.StagedPassManager)を生成します。各ステージを構成する具体的なパスは、`generate_preset_pass_manager` に渡す引数によって異なります。`optimization_level` は必須の位置引数であり、0、1、2、3 のいずれかの整数で指定します。値が大きいほど最適化が強力になりますが、計算コストも高くなります（[トランスパイルのデフォルト設定と構成オプション](defaults-and-configuration-options)を参照）。

回路をトランスパイルする推奨の方法は、プリセットのステージ付きパスマネージャーを作成し、そのパスマネージャーを回路に対して実行することです。詳細は[パスマネージャーによるトランスパイル](transpile-with-pass-managers)を参照してください。ただし、カスタマイズ性は低くなりますが、よりシンプルな代替手段として [`transpile`](https://docs.quantum.ibm.com/api/qiskit/compiler#qiskit.compiler.transpile) 関数を使用する方法もあります。この関数は回路を直接引数として受け取ります。`generate_preset_pass_manager` と同様に、使用されるトランスパイラーパスは `optimization_level` などの引数によって決まります。実際、内部では `transpile` 関数が `generate_preset_pass_manager` を呼び出してプリセットのステージ付きパスマネージャーを生成し、それを回路に対して実行しています。

## Init ステージ
最初のステージは、デフォルトではほとんど何も行いません。独自の初期最適化を追加したい場合に主に活用されます。また、多くのレイアウトおよびルーティングアルゴリズムは単一・二量子ビットゲートのみを対象として設計されているため、このステージでは三量子ビット以上のゲートを一・二量子ビットゲートに変換する処理も行われます。

このステージに独自の初期最適化を実装する方法については、プラグインおよびパスマネージャーのカスタマイズに関するセクションを参照してください。

## Layout ステージ
次のステージは、回路を送信するバックエンドのレイアウト（接続構造）に関するものです。一般に、量子回路は抽象的な存在であり、その量子ビットは実際の計算に使用される物理的な量子ビットの「仮想」または「論理的」な表現です。ゲートのシーケンスを実行するには、「仮想」量子ビットと実際の量子デバイスの「物理」量子ビットとの一対一の対応付けが必要です。このマッピングは `Layout` オブジェクトとして保持され、バックエンドの[命令セットアーキテクチャ（ISA）](./transpile#instruction-set-architecture)で定義される制約の一部です。

![この画像は、ワイヤー表現の量子ビットが QPU 上の接続を表す図にマッピングされる様子を示しています。](../docs/images/guides/transpiler-stages/layout-mapping.avif "Qubit mapping")

マッピングの選択は、入力回路をデバイスのトポロジーにマッピングするために必要な SWAP 操作の数を最小化し、最もキャリブレーションの良い量子ビットを使用するうえで非常に重要です。このステージの重要性から、プリセットパスマネージャーは最適なレイアウトを見つけるためにいくつかの異なる手法を試みます。通常、これは二つのステップで構成されます。まず「完全な」レイアウト（SWAP 操作が不要なレイアウト）を見つけようとし、次に完全なレイアウトが見つからない場合に最適なレイアウトを探すヒューリスティックパスを実行します。最初のステップには、通常以下の二つの `Pass` が使用されます。

- `TrivialLayout`: 各仮想量子ビットをデバイスの同じ番号の物理量子ビットに単純にマッピングします（例：[`0`,`1`,`1`,`3`] -> [`0`,`1`,`1`,`3`]）。これは `optimization_level=1` でのみ完全なレイアウト探索に使用される歴史的な動作です。失敗した場合は次に `VF2Layout` が試みられます。
- `VF2Layout`: このステージをサブグラフ同型問題として扱い、VF2++ アルゴリズムで解く `AnalysisPass` です。複数のレイアウトが見つかった場合は、スコアリングヒューリスティックを実行して平均エラーが最も低いマッピングを選択します。

ヒューリスティックステージでは、デフォルトで以下の二つのパスが使用されます。

- `DenseLayout`: 回路と同じ量子ビット数を持つデバイスのサブグラフのうち、接続性が最も高いものを探します（回路に `IfElseOp` などの制御フロー操作が含まれている場合、最適化レベル 1 で使用されます）。
- `SabreLayout`: ランダムな初期レイアウトから出発し、`SabreSwap` アルゴリズムを繰り返し実行することでレイアウトを選択するパスです。`VF2Layout` パスで完全なレイアウトが見つからない場合に、最適化レベル 1、2、3 でのみ使用されます。このアルゴリズムの詳細については、論文 [arXiv:1809.02573](https://arxiv.org/abs/1809.02573) を参照してください。

## Routing ステージ
量子デバイス上で直接接続されていない量子ビット間の二量子ビットゲートを実行するには、回路に一つ以上の SWAP ゲートを挿入して、量子ビットの状態をデバイスのゲートマップ上で隣接するまで移動させる必要があります。各 SWAP ゲートは高コストでノイズの多い操作です。したがって、回路を所定のデバイスにマッピングするために必要な SWAP ゲートの最小数を求めることは、トランスパイルプロセスにおける重要なステップです。効率化のため、このステージはデフォルトで Layout ステージと並行して計算されますが、論理的には両者は独立しています。*Layout* ステージは使用するハードウェア量子ビットを選択し、*Routing* ステージは選択したレイアウトで回路を実行するために必要な SWAP ゲートを適切な数だけ挿入します。

しかし、最適な SWAP マッピングを求めることは困難です。実際、これは NP 困難問題であり、ごく小さな量子デバイスや入力回路を除いて計算コストが非常に高くなります。これを回避するため、Qiskit では `SabreSwap` という確率的なヒューリスティックアルゴリズムを使用して、最適ではないかもしれませんが良質な SWAP マッピングを計算します。確率的な手法を使用するため、生成される回路は実行ごとに同一になるとは限りません。実際、同じ回路を繰り返し実行すると、出力の回路深さとゲート数が分布として現れます。そのため、多くのユーザーはルーティング関数（または `StagedPassManager` 全体）を何度も実行し、出力の分布から最も深さの小さい回路を選択することを選びます。

例として、15 量子ビットの GHZ 回路を、「悪い」（非接続な）`initial_layout` を使用して 100 回実行してみましょう。

In [1]:
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit
from qiskit.transpiler import generate_preset_pass_manager
from qiskit.providers.fake_provider import GenericBackendV2

backend = GenericBackendV2(15)


ghz = QuantumCircuit(15)
ghz.h(0)
ghz.cx(0, range(1, 15))

depths = []
for seed in range(100):
    pass_manager = generate_preset_pass_manager(
        optimization_level=1,
        backend=backend,
        layout_method="trivial",  # Fixed layout mapped in circuit order
        seed_transpiler=seed,  # For reproducible results
    )
    depths.append(pass_manager.run(ghz).depth())

plt.figure(figsize=(8, 6))
plt.hist(depths, align="left", color="#AC557C")
plt.xlabel("Depth", fontsize=14)
plt.ylabel("Counts", fontsize=14)

Text(0, 0.5, 'Counts')

<Image src="../docs/images/guides/transpiler-stages/extracted-outputs/62479697-cef1-4a89-ac2a-20051dd294f4-1.svg" alt="Output of the previous code cell" />

This wide distribution demonstrates how difficult it is for the SWAP mapper to compute the best mapping.  To gain some insight, let's look at both the circuit being executed as well as the qubits that were chosen on the hardware.

In [2]:
ghz.draw("mpl", idle_wires=False)

<Image src="../docs/images/guides/transpiler-stages/extracted-outputs/ab89c4ea-06c4-4320-b493-feb691b3570d-0.svg" alt="Output of the previous code cell" />

In [3]:
from qiskit.visualization import plot_circuit_layout

# Plot the hardware graph and indicate which hardware qubits were chosen to run the circuit
transpiled_circ = pass_manager.run(ghz)
plot_circuit_layout(transpiled_circ, backend)

<Image src="../docs/images/guides/transpiler-stages/extracted-outputs/f6a3a92a-8656-4518-ba2c-c3b0b038f507-0.svg" alt="Output of the previous code cell" />

As you can see, this circuit has to execute a two-qubit gate between qubits 0 and 14, which are very far apart on the connectivity graph.  Running this circuit thus requires inserting SWAP gates to execute all of the two-qubit gates using the `SabreSwap` pass.

Note also that the `SabreSwap` algorithm is different from the larger `SabreLayout` method in the previous stage.  By default, `SabreLayout` runs both layout and routing, and returns the transformed circuit.  This is done for a few particular technical reasons specified in the pass's [API reference page](../api/qiskit/qiskit.transpiler.passes.SabreLayout).

## Translation stage

When writing a quantum circuit, you are free to use any quantum gate (unitary operation) that you like, along with a collection of non-gate operations such as qubit measurement or reset instructions.  However, most quantum devices only natively support a handful of quantum gate and non-gate operations.  These native gates are part of the definition of a target's [ISA](/docs/guides/transpile#instruction-set-architecture) and this stage of the preset `PassManagers`  translates (or *unrolls*) the gates specified in a circuit to the native basis gates of a specified backend.  This is an important step, as it allows the circuit to be executed by the backend, but typically leads to an increase in the depth and number of gates.

Two special cases are especially important to highlight, and help illustrate what this stage does.

1. If a SWAP gate is not a native gate to the target backend, this requires three CNOT gates:

In [4]:
print("native gates:" + str(sorted(backend.operation_names)))
qc = QuantumCircuit(2)
qc.swap(0, 1)
qc.decompose().draw("mpl")

native gates:['cx', 'delay', 'id', 'measure', 'reset', 'rz', 'sx', 'x']


<Image src="../docs/images/guides/transpiler-stages/extracted-outputs/d4d1f65a-3336-4d70-9189-65ba010f2366-1.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/transpiler-stages/extracted-outputs/e4cd49ef-5b3e-4ee0-82f8-c83e1f0ae755-0.svg)

ご覧のとおり、この回路は接続グラフ上で非常に遠い位置にある量子ビット 0 と 14 の間で二量子ビットゲートを実行する必要があります。そのため、この回路を実行するには `SabreSwap` パスですべての二量子ビットゲートを実行できるよう SWAP ゲートを挿入する必要があります。

なお、`SabreSwap` アルゴリズムは前のステージで使用した `SabreLayout` メソッドとは異なるものであることにも注意してください。デフォルトでは、`SabreLayout` はレイアウトとルーティングの両方を実行し、変換後の回路を返します。これはパスの [API リファレンスページ](../api/qiskit/qiskit.transpiler.passes.SabreLayout)に記載されているいくつかの技術的な理由によるものです。

## Translation ステージ
量子回路を記述する際は、任意の量子ゲート（ユニタリ操作）を自由に使用できます。また、量子ビットの測定やリセット命令などのゲート以外の操作も利用できます。ただし、ほとんどの量子デバイスがネイティブにサポートするのは、ごく少数の量子ゲートおよびゲート以外の操作に限られています。これらのネイティブゲートはターゲットの [ISA](./transpile#instruction-set-architecture) の定義の一部であり、このプリセット `PassManagers` のステージは、回路に指定されたゲートを指定されたバックエンドのネイティブ基底ゲートに変換（*アンロール*）します。これは回路をバックエンドで実行するための重要なステップですが、通常は深さとゲート数の増加を招きます。

特に重要な二つの特殊なケースを取り上げて、このステージが何を行うかを説明します。

1. SWAP ゲートがターゲットバックエンドのネイティブゲートでない場合、三つの CNOT ゲートが必要です。

In [5]:
qc = QuantumCircuit(3)
qc.ccx(0, 1, 2)
qc.decompose().draw("mpl")

<Image src="../docs/images/guides/transpiler-stages/extracted-outputs/4552b367-75e9-4faa-a59d-8317e25ac145-0.svg" alt="Output of the previous code cell" />

For every Toffoli gate in a quantum circuit, the hardware may execute up to six CNOT gates and a handful of single-qubit gates.  This example demonstrates that any algorithm making use of multiple Toffoli gates will end up as a circuit with large depth and will therefore be appreciably affected by noise.

## Optimization stage

This stage centers around decomposing quantum circuits into the basis gate set of the target device, and must fight against the increased depth from the layout and routing stages.  Fortunately, there are many routines for optimizing circuits by either combining or eliminating gates.  In some cases, these methods are so effective that the output circuits have lower depth than the inputs, even after layout and routing to the hardware topology.  In other cases, not much can be done, and the computation may be difficult to perform on noisy devices.  This stage is where the various optimization levels begin to differ.

- For `optimization_level=1`, this stage prepares [`Optimize1qGatesDecomposition`](../api/qiskit/qiskit.transpiler.passes.Optimize1qGatesDecomposition) and [`CXCancellation`](/docs/api/qiskit/1.4/qiskit.transpiler.passes.CXCancellation), which combine chains of single-qubit gates and cancel any back-to-back CNOT gates.
- For `optimization_level=2`, this stage uses the [`CommutativeCancellation`](../api/qiskit/qiskit.transpiler.passes.CommutativeCancellation) pass instead of `CXCancellation`, which removes redundant gates by exploiting commutation relations.
- For `optimization_level=3`, this stage prepares the following passes:
  - [`Collect2qBlocks`](../api/qiskit/qiskit.transpiler.passes.Collect2qBlocks)
  - [`ConsolidateBlocks`](../api/qiskit/qiskit.transpiler.passes.ConsolidateBlocks)
  - [`UnitarySynthesis`](../api/qiskit/qiskit.transpiler.passes.UnitarySynthesis)
  - [`Optimize1qGateDecomposition`](../api/qiskit/qiskit.transpiler.passes.Optimize1qGatesDecomposition)
  - [`CommutativeCancellation`](../api/qiskit/qiskit.transpiler.passes.CommutativeCancellation)


Additionally, this stage also executes a few final checks to make sure that all instructions in the circuit are composed of the basis gates available on the target backend.

The example below using a GHZ state demonstrates the effects of different optimization level settings on circuit depth and gate count.

<Admonition type="note">
  The transpilation output varies due to the stochastic SWAP mapper. Therefore, the numbers below will likely change each time you run the code.
</Admonition>

![15-qubit GHZ state](../docs/images/guides/transpiler-stages/transpiler-11.avif "15-qubit GHZ state before transpilation")

The following code constructs a 15-qubit GHZ state and compares the `optimization_levels` of transpilation in terms of resulting circuit depth, gate counts, and multi-qubit gate counts.

In [6]:
ghz = QuantumCircuit(15)
ghz.h(0)
ghz.cx(0, range(1, 15))

depths = []
gate_counts = []
multiqubit_gate_counts = []
levels = [str(x) for x in range(4)]
for level in range(4):
    pass_manager = generate_preset_pass_manager(
        optimization_level=level,
        backend=backend,
        seed_transpiler=1234,
    )
    circ = pass_manager.run(ghz)
    depths.append(circ.depth())
    gate_counts.append(sum(circ.count_ops().values()))
    multiqubit_gate_counts.append(circ.count_ops()["cx"])

fig, (ax1, ax2) = plt.subplots(2, 1)
ax1.bar(levels, depths, label="Depth")
ax1.set_xlabel("Optimization Level")
ax1.set_ylabel("Depth")
ax1.set_title("Output Circuit Depth")
ax2.bar(levels, gate_counts, label="Number of Circuit Operations")
ax2.bar(levels, multiqubit_gate_counts, label="Number of CX gates")
ax2.set_xlabel("Optimization Level")
ax2.set_ylabel("Number of gates")
ax2.legend()
ax2.set_title("Number of output circuit gates")
fig.tight_layout()
plt.show()

<Image src="../docs/images/guides/transpiler-stages/extracted-outputs/2700405a-f559-45d3-99a9-5b4447621743-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/transpiler-stages/extracted-outputs/31fcf8b6-4f3a-460e-90fd-446813bd4f28-1.svg)

SWAP は三つの CNOT ゲートの積であるため、ノイズの多い量子デバイスでは高コストな操作です。しかし、多くのデバイスが持つ限られたゲート接続性に回路を埋め込むためには、通常こうした操作が必要になります。したがって、回路内の SWAP ゲートの数を最小化することがトランスパイルプロセスの主要な目標の一つです。

2. Toffoli ゲート、すなわち controlled-controlled-not ゲート（`ccx`）は三量子ビットゲートです。基底ゲートセットに単一・二量子ビットゲートしか含まれていない場合、このゲートを分解する必要があります。ただし、そのコストはかなり大きくなります。

In [7]:
ghz = QuantumCircuit(5)
ghz.h(0)
ghz.cx(0, range(1, 5))


# Use fake backend
backend = GenericBackendV2(5)

# Run with optimization level 3 and 'asap' scheduling pass
pass_manager = generate_preset_pass_manager(
    optimization_level=3,
    backend=backend,
    scheduling_method="asap",
    seed_transpiler=1234,
)


circ = pass_manager.run(ghz)
circ.draw(output="mpl", idle_wires=False)

<Image src="../docs/images/guides/transpiler-stages/extracted-outputs/ae2d9390-5c26-46f3-9418-a684ba8a406a-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/transpiler-stages/extracted-outputs/8a2c1913-3324-44b0-859e-d7fd348161c3-0.svg)

量子回路内の Toffoli ゲート一つにつき、ハードウェアは最大で六つの CNOT ゲートといくつかの単一量子ビットゲートを実行する可能性があります。この例は、複数の Toffoli ゲートを使用するアルゴリズムは深さの大きな回路になり、ノイズの影響を大きく受けることを示しています。

## Optimization ステージ
このステージは、量子回路をターゲットデバイスの基底ゲートセットに分解することを中心としており、Layout ステージと Routing ステージによる深さの増加に対抗する必要があります。幸い、ゲートを結合または除去することで回路を最適化する手法が数多く存在します。場合によっては、これらの手法が非常に効果的であり、レイアウトおよびルーティング後でも出力回路の深さが入力より小さくなることがあります。一方、最適化の余地が少なくノイズの多いデバイスでの実行が困難なケースもあります。このステージで各最適化レベルの違いが現れ始めます。

- `optimization_level=1` の場合、このステージは [`Optimize1qGatesDecomposition`](../api/qiskit/qiskit.transpiler.passes.Optimize1qGatesDecomposition) と [`CXCancellation`](https://docs.quantum.ibm.com/api/qiskit/1.4/qiskit.transpiler.passes.CXCancellation) を用意し、単一量子ビットゲートの連鎖を結合し、連続する CNOT ゲートをキャンセルします。
- `optimization_level=2` の場合、このステージは `CXCancellation` の代わりに [`CommutativeCancellation`](../api/qiskit/qiskit.transpiler.passes.CommutativeCancellation) パスを使用し、可換性を活用して冗長なゲートを削除します。
- `optimization_level=3` の場合、このステージは以下のパスを用意します。
  - [`Collect2qBlocks`](../api/qiskit/qiskit.transpiler.passes.Collect2qBlocks)
  - [`ConsolidateBlocks`](../api/qiskit/qiskit.transpiler.passes.ConsolidateBlocks)
  - [`UnitarySynthesis`](../api/qiskit/qiskit.transpiler.passes.UnitarySynthesis)
  - [`Optimize1qGateDecomposition`](../api/qiskit/qiskit.transpiler.passes.Optimize1qGatesDecomposition)
  - [`CommutativeCancellation`](../api/qiskit/qiskit.transpiler.passes.CommutativeCancellation)

さらに、このステージでは回路内のすべての命令がターゲットバックエンドで利用可能な基底ゲートで構成されているかどうかを確認する最終チェックも実行されます。

以下に示す GHZ 状態の例は、最適化レベルの設定が回路深さとゲート数に与える影響を示しています。

> **Note:** トランスパイルの出力は確率的な SWAP マッパーによって異なります。したがって、以下の数値はコードを実行するたびに変わる可能性があります。

![15 量子ビットの GHZ 状態](../docs/images/guides/transpiler-stages/transpiler-11.avif "トランスパイル前の 15 量子ビット GHZ 状態")

以下のコードは 15 量子ビットの GHZ 状態を構築し、トランスパイルの `optimization_level` ごとに、結果として得られる回路の深さ、ゲート数、マルチ量子ビットゲート数を比較しています。